# 五会话标签顺序回放运行预算

冻结2019-12-19的最早40个来源合法分钟，仅量单事件引擎耗时及现金/整股/终端NAV身份；不以未来价值选事件。若外推完整382996条成熟路径超过30分钟，顺序引擎不批量跑，改成同规则按日向量化并另核单事件一致性。

In [1]:
import os,json,hashlib,warnings
from pathlib import Path
os.environ["MLFLOW_ALLOW_FILE_STORE"]="true"
os.environ["MLFLOW_DISABLE_AGENT_HINT"]="1"
warnings.filterwarnings("ignore",message="The 'generic' unit for NumPy timedelta")
import numpy as np
import pandas as pd
import qlib
from qlib.data import D
from src.core.recorder_sources import load_verified_recorder_objects
from src.backtest.stage1_event_checkpoint import capture_after_open
from src.backtest.stage1_capfree_lock_episode import replay_capfree_lock_episode
from src.backtest.stage1_capfree_lock_pilot import replay_capfree_lock_pilot
from src.backtest.ibkr_tiered import settle_tiered_event
from src.workflow.stage1_dated_controller import DatedModel,DatedStage1Controllers
from src.workflow.stage1_f696_training_reference import prepare as prepare_f696_source
from src.workflow.intraday_recovery_direct_allocation import IntradayRecoveryDirectAllocationSpec
from src.workflow.stage1_open_action_value import Stage1OpenActionValueSpec
from copy import deepcopy
root=Path.cwd();card=root/"research/stage1_v5_lock_label_runtime_20261030";plan=json.loads((card/"plan.json").read_text())
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
parent=root/plan["parent_card"];old=root/plan["old_source_card"]
assert sha(parent/"summary.json")==plan["parent_summary_sha256"] and sha(parent/"verification_manifest.json")==plan["parent_manifest_sha256"]
assert sha(old/"plan.json")==plan["old_source_plan_sha256"]
assert sha(root/plan["checkpoint_code"])==plan["checkpoint_code_sha256"]
assert sha(root/plan["episode_module"])==plan["episode_module_sha256"]
assert sha(root/plan["source_support_card"]/"summary.json")==plan["source_support_summary_sha256"]
assert sha(root/plan["minute_provider_uri"]/"qlib_store_manifest.json")==plan["minute_provider_manifest_sha256"]
parent_plan=json.loads((parent/"plan.json").read_text());old_plan=json.loads((old/"plan.json").read_text())
prepared=prepare_f696_source(root=root,card=old,plan=old_plan)
qlib.init(provider_uri=str(root/plan["minute_provider_uri"]),region="us",kernels=1)
names=["analysis/native/dated_f696_0","analysis/controller_audit",*[f"models/open_{y}" for y in parent_plan["signal_years"]],*[f"models/recovery_{y}" for y in (2020,2021,2022)]]
objects,receipt=load_verified_recorder_objects(parent,expected_recorder_id=plan["parent_recorder_id"],artifacts=names,expected_manifest_sha256=plan["parent_manifest_sha256"])
book=objects["analysis/native/dated_f696_0"];frame=prepared["frame"]
assert len(frame)==1056 and book["audit"]["status"]=="passed"
open_models={y:DatedModel(objects[f"models/open_{y}"],f"{y-1}-12-31",f"{parent_plan['action_recorder_id']}:open_action_value_models/{y}") for y in parent_plan["signal_years"]}
recovery_models={}
for y in parent_plan["signal_years"]:
 if y in parent_plan["explicit_no_recovery_model_years"]:recovery_models[y]=DatedModel(None,f"{y-1}-12-31",f"explicit_no_model_{y}_insufficient_prior_teacher",explicit_no_model=True)
 else:recovery_models[y]=DatedModel(objects[f"models/recovery_{y}"],f"{y-1}-12-31",f"{plan['parent_recorder_id']}:models/recovery_{y}")
controller=DatedStage1Controllers(calendar=frame[["signal_date","trade_date"]],open_models=open_models,recovery_models=recovery_models)
pd.testing.assert_frame_equal(controller.audit(),objects["analysis/controller_audit"],check_dtype=False)
recovery_spec=IntradayRecoveryDirectAllocationSpec(root=root,**prepared["action_plan"]["recovery_spec"])
action_spec=Stage1OpenActionValueSpec(root=root,**prepared["action_plan"]["action_spec"])
print('dated parent and minute source verified',len(receipt))

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[27480:MainThread](2026-10-30 14:06:40,242) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[27480:MainThread](2026-10-30 14:06:40,243) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[27480:MainThread](2026-10-30 14:06:40,244) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/full_path_tiered_allocation_20261013')}


[27480:MainThread](2026-10-30 14:06:40,315) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[27480:MainThread](2026-10-30 14:06:40,316) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[27480:MainThread](2026-10-30 14:06:40,316) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/stage1_event_guard_fullminute_20261028')}


dated parent and minute source verified 13


In [2]:
import time
from src.backtest.event_guard_clock import guard_slot
day=pd.Timestamp(plan['event_day']);checkpoint=capture_after_open(frame,event_day=day,controller_by_date=controller,recovery_spec=recovery_spec,action_spec=action_spec,selection=prepared['selection'],plan=parent_plan)
last=frame.iloc[checkpoint.index+4]
rawbars=D.features(['SOXL','SOXX'],['$open','$close','$raw_open','$price_valid'],start_time=str(day.date())+' 09:31:00',end_time=str(pd.Timestamp(last.trade_date).date())+' 16:00:00',freq='1min')
parts=[]
for symbol in ('SOXL','SOXX'):
 x=rawbars.xs(symbol,level='instrument').copy();x.columns=[symbol+'_'+k.removeprefix('$') for k in x.columns];parts.append(x)
quotes=pd.concat(parts,axis=1).sort_index()
assert quotes[['SOXL_price_valid','SOXX_price_valid']].eq(1).all().all()
first=quotes.loc[quotes.index.normalize()==day]
clocks=first.index[(first.index>=day+pd.Timedelta(hours=9,minutes=48)) & (first.index<day+pd.Timedelta(hours=10,minutes=29))][:plan['sample_events']]
assert len(clocks)==plan['sample_events'] and len(set(clocks))==len(clocks)
source_day=pd.read_csv(root/plan['source_support_card']/'perday_support.csv');source_row=source_day.loc[source_day.trade_date.eq(str(day.date()))].iloc[0]
assert source_row.paired_funded>=plan['sample_events']
print('preselected causal clocks',str(clocks[0]),str(clocks[-1]))

preselected causal clocks 2019-12-19 09:48:00 2019-12-19 10:27:00


In [3]:
results=[];start=time.perf_counter()
for stamp in clocks:
 mark=first.loc[stamp];price=np.array([mark.SOXL_close,mark.SOXX_close],float)
 nav=float(checkpoint.state.q@price+checkpoint.state.cash);w=np.r_[checkpoint.state.q*price,checkpoint.state.cash]/nav
 target=w+np.array([0.,.10,-.10])
 slot=guard_slot(stamp,session_open=day+pd.Timedelta(hours=9,minutes=30),session_close=first.index.max(),original_open_fill=day+pd.Timedelta(hours=9,minutes=47))
 assert slot is not None
 bar=first.loc[slot.fill_at+pd.Timedelta(minutes=1)]
 event={'trade_date':day,'bar_end_at':stamp,'decision_at':slot.decision_at,'fill_at':slot.fill_at,'desired_weights':target,'adjusted_fill_prices':np.array([bar.SOXL_open,bar.SOXX_open],float),'raw_fill_prices':np.array([bar.SOXL_raw_open,bar.SOXX_raw_open],float)}
 out=replay_capfree_lock_episode(frame,checkpoint,quotes,event=event,horizon_sessions=5)
 assert out['event_status']=='funded' and len(out['EVENT']['orders'])==1
 last_price=last[['soxl_exit_open','soxx_exit_open']].to_numpy(float)
 for arm in ('HOLD','EVENT'):
  v=out[arm];assert abs(float(v['ending_quantities']@last_price+v['ending_cash'])-float(v['path'].iloc[-1].next_open_nav))<1e-7
 results.append(out)
elapsed=time.perf_counter()-start
first_event=results[0];first_stamp=clocks[0];first_bar=first.loc[first_stamp];price=np.array([first_bar.SOXL_close,first_bar.SOXX_close],float)
nav=float(checkpoint.state.q@price+checkpoint.state.cash);w=np.r_[checkpoint.state.q*price,checkpoint.state.cash]/nav
slot=guard_slot(first_stamp,session_open=day+pd.Timedelta(hours=9,minutes=30),session_close=first.index.max(),original_open_fill=day+pd.Timedelta(hours=9,minutes=47))
bar=first.loc[slot.fill_at+pd.Timedelta(minutes=1)]
legacy_event={'trade_date':day,'bar_end_at':first_stamp,'decision_at':slot.decision_at,'fill_at':slot.fill_at,'desired_weights':w+np.array([0.,.10,-.10]),'adjusted_fill_prices':np.array([bar.SOXL_open,bar.SOXX_open],float),'raw_fill_prices':np.array([bar.SOXL_raw_open,bar.SOXX_raw_open],float)}
legacy=replay_capfree_lock_pilot(frame,checkpoint,quotes,event=legacy_event,horizon_sessions=5)
terminal_errors={arm:abs(float(first_event[arm]['path'].iloc[-1].next_open_nav)-float(legacy[arm]['path'].iloc[-1].next_open_nav)) for arm in ('HOLD','EVENT')}
assert max(terminal_errors.values())<1e-7
projected=elapsed/len(clocks)*plan['full_mature_path_rows']
audit={'experiment':plan['experiment'],'plan_sha256':sha(card/'plan.json'),'episode_module_sha256':sha(root/plan['episode_module']),'sample_events':len(clocks),'sample_day':str(day.date()),'elapsed_seconds':elapsed,'seconds_per_event':elapsed/len(clocks),'naive_full_path_projection_seconds':projected,'maximum_budget_seconds':plan['maximum_projected_seconds_for_full_mature_path_rows'],'sequential_full_run_admitted':bool(projected<=plan['maximum_projected_seconds_for_full_mature_path_rows']),'legacy_first_event_terminal_nav_errors_usd':terminal_errors,'all_sample_event_orders_one':True,'new_model_fits':0,'new_candidate_accounts':0,'historical_research_passed':False,'decision':'runtime_projection_requires_vectorization' if projected>plan['maximum_projected_seconds_for_full_mature_path_rows'] else 'sequential_runtime_within_budget'}
(card/'audit.json').write_text(json.dumps(audit,ensure_ascii=False,indent=2)+'\n')
print(json.dumps(audit,ensure_ascii=False,indent=2))

{
  "experiment": "stage1_v5_lock_label_runtime_20261030",
  "plan_sha256": "08814e85b6eb78d14ea948001f2797bb6356c9e8c3b3d1635a46b702c6752e6e",
  "episode_module_sha256": "8654a7c8adff7d5eff67013e24b0b299eb24c2d5cd5175c8676ba4787672819d",
  "sample_events": 40,
  "sample_day": "2019-12-19",
  "elapsed_seconds": 0.4269927090208512,
  "seconds_per_event": 0.01067481772552128,
  "naive_full_path_projection_seconds": 4088.412489603748,
  "maximum_budget_seconds": 1800,
  "sequential_full_run_admitted": false,
  "legacy_first_event_terminal_nav_errors_usd": {
    "HOLD": 0.0,
    "EVENT": 0.0
  },
  "all_sample_event_orders_one": true,
  "new_model_fits": 0,
  "new_candidate_accounts": 0,
  "historical_research_passed": false,
  "decision": "runtime_projection_requires_vectorization"
}


## 中文结论

SOXX：零新拟合/账户。第一次时钟预检因09:48至10:29不足50根分钟bar而在运行时间/价值读取前停止；原计划和失败收据保留，仅按钟表改成同日最早40根，实际Qlib读2次。顺序五会话锁定回放40事件耗时约0.427秒，平均10.675毫秒/事件；对382996成熟来源行的线性工程外推约4088秒，超过事前1800秒预算。首事件HOLD/EVENT终端NAV与旧机械pilot均误差0；没有根据未来收益选时刻。决定不直接启动顺序全量，改按日编译时间有序价格平面并精确对账。2021/2022价值未访问。